# 第 8 课：连接外部世界——工具人体工学、MCP 通用插座与 Coding Agent
### 《现代 AI 与 Agent 系统》Colab 实战实验 · 走向外部真实环境

> **为什么需要这节课？**
> 前面 7 课我们学会了控制流、状态持久化和分层记忆，但 Agent 终究要迈出自身容器，去**操作文件、查询数据库、调用 API、甚至自主阅读与修复代码**！
> 如果工具接口设计得像给人类看的传统 REST API 一样碎粒度，模型不仅会陷入大量无效网络往返，还极易填错参数；同时，如何解除对单一框架的绑定，实现通用工具即插即用（Model Context Protocol, MCP）？如何实现从定位问题到跑测试验证的自动化 Coding 闭环？

---

### 本课 3 个核心动手实验（先写预测 $	o$ 运行实测 $	o$ 用数字说话）
1. **实验 1（工具人体工学对比）**：对比“碎片化 API（查用户 -> 查订单 -> 查优惠 -> 计算，需 4~6 轮往返且易错）”vs“面向结果的粗粒度工具（1 轮完成，Token 节省 65%）”；
2. **实验 2（MCP 工具池按需搜索）**：对比“一次性把 50 个工具 Schema 全部塞入上下文”vs“两阶段动态发现（Code Mode）”的上下文消耗；
3. **实验 3（Coding Agent 读-改-验闭环）**：模拟 Agent 在真实 Python 脚本上执行 `Understand -> Locate -> Modify -> Verify`，自动跑通单元测试并输出防伪凭证！


In [ ]:
# [Cell 1: 环境初始化与工具沙箱模拟器]
import copy
import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

print("[OK] Cell 1 初始化完成！外部工具工程与 Coding Agent 沙箱已就绪。")


---
## 实验 1：工具人体工学（Agent Ergonomics）——碎粒度 API vs 结果导向工具

### 1. 为什么需要它？
很多初学者把现有的微服务 REST API 直接原样包成 Tool 扔给 Agent。例如为了“退款并注销”，要求 Agent 依次调用 `get_user`, `list_orders`, `check_eligibility`, `calculate_penalty`, `execute_refund`, `send_sms`。
大模型不仅要在 6 轮往返中重复阅读整个对话历史，任何一步出微小差错，整个任务直接断裂。

### 2. 预测与实测
对比完成同一项退款任务：
- **做法 A（细粒度 API 串联）**：总往返轮数与累计消耗 Token 量；
- **做法 B（面向结果的粗粒度工具 `process_account_closure_refund`）**：总往返轮数与 Token 量。


In [ ]:
# [Cell 2: 实验 1 — 细粒度工具 vs 面向结果工具实测对比]

# 模拟细粒度 API 往返
def simulate_fine_grained_api_flow() -> Dict[str, Any]:
    steps = ["get_user", "list_orders", "check_eligibility", "execute_refund", "send_sms"]
    turns = len(steps)
    tokens_per_turn = [1200 + i * 400 for i in range(turns)]  # 历史不断膨胀
    total_tokens = sum(tokens_per_turn)
    return {"turns": turns, "total_tokens": total_tokens, "error_risk_pct": 26.5}

# 模拟面向结果的粗粒度工具（Outcome-oriented Tool）
def simulate_outcome_oriented_tool_flow() -> Dict[str, Any]:
    turns = 1
    total_tokens = 1500  # 单次请求直接完成
    return {"turns": turns, "total_tokens": total_tokens, "error_risk_pct": 3.2}

res_fine = simulate_fine_grained_api_flow()
res_outcome = simulate_outcome_oriented_tool_flow()

print("=== 实验 1 实测结果（退款业务调用对比）===")
print(f"做法 A (细粒度 API 包装)     : 往返轮数 = {res_fine['turns']} 轮 | 累计 Token = {res_fine['total_tokens']:,} | 级联出错风险 = {res_fine['error_risk_pct']}%")
print(f"做法 B (结果导向粗粒度工具) : 往返轮数 = {res_outcome['turns']} 轮 | 累计 Token = {res_outcome['total_tokens']:,} | 级联出错风险 = {res_outcome['error_risk_pct']}%")
print(f"💡 结论：结果导向工具让 Token 消耗直降 {(1 - res_outcome['total_tokens']/res_fine['total_tokens'])*100:.1f}%，往返交互减少 80%！")


---
## 实验 2：Model Context Protocol (MCP) 与大规模工具按需搜索

### 1. 为什么需要它？
当一个企业 Agent 接入了 50 个微服务（数据库、CRM、邮件、Kubernetes、AWS 等），如果把 50 个工具的完整 JSON Schema 全部塞进每一个请求：
- 仅工具描述就要占去 **15,000~20,000 Tokens** 的宝贵窗口！
- 工具之间功能重叠，大模型极容易选错工具名称。
- **MCP 两阶段发现（Progressive Tool Discovery）**：开局只给工具名与一行简短功能索引（仅需 800 Tokens）；模型提出意图后，按需检索并挂载具体工具 Schema！


In [ ]:
# [Cell 3: 实验 2 — 全量工具注入 vs MCP 两阶段按需加载]

TOOL_CATALOG = {f"tool_{i:02d}": f"Detailed JSON schema with 10 parameters for service {i}..." for i in range(50)}
TOOL_INDEX = {f"tool_{i:02d}": f"Service {i} operations." for i in range(50)}

def measure_tool_tokens_eager() -> int:
    # 全量工具描述 Token 估算（每个工具约 350 tokens）
    return len(TOOL_CATALOG) * 350

def measure_tool_tokens_mcp_lazy(needed_tools: List[str]) -> int:
    # 简短索引 (每个 15 tokens) + 仅加载用到的工具详细 Schema (每个 350 tokens)
    index_tokens = len(TOOL_INDEX) * 15
    loaded_tokens = len(needed_tools) * 350
    return index_tokens + loaded_tokens

tokens_eager = measure_tool_tokens_eager()
tokens_lazy = measure_tool_tokens_mcp_lazy(["tool_03", "tool_18"])

print("=== 实验 2 实测结果（50 个工具池治理）===")
print(f"1. 全量静态加载 (Eager) : 工具定义占用 = {tokens_eager:,} tokens")
print(f"2. MCP 两阶段发现 (Lazy) : 工具定义占用 = {tokens_lazy:,} tokens (仅加载 2 个所需工具)")
print(f"💡 节省上下文空间: {(1 - tokens_lazy / tokens_eager)*100:.1f}%！")


---
## 实验 3：Coding Agent 自动化实战——读-改-验闭环（Understand -> Locate -> Modify -> Verify）

### 1. 为什么需要它？
Coding Agent 最核心的纪律是：**绝不轻信模型自称的“我已修好”，宿主必须在隔离沙箱中运行真实单元测试，绿灯才算真正完成！**


In [ ]:
# [Cell 4: 实验 3 — Coding Agent 自动化 Bug 修复闭环]

SOURCE_CODE_BUGGY = '''def calculate_discount(price: float, is_vip: bool) -> float:
    # BUG: VIP 应该是 8 折 (0.8)，这里写成了 0.9
    discount = 0.9 if is_vip else 1.0
    return price * discount
'''

def run_tests(code_str: str) -> bool:
    env = {}
    try:
        exec(code_str, env)
        fn = env["calculate_discount"]
        # Test case 1: normal user
        assert fn(100.0, False) == 100.0
        # Test case 2: VIP user (should be 80.0)
        assert fn(100.0, True) == 80.0
        return True
    except Exception:
        return False

# 模拟 Coding Agent 闭环
def run_coding_agent_loop() -> Tuple[bool, str]:
    # 1. Locate: 模型识别到 discount = 0.9 处错误
    # 2. Modify: 模型提出 Search/Replace 补丁
    patched_code = SOURCE_CODE_BUGGY.replace("0.9 if is_vip", "0.8 if is_vip")
    # 3. Verify: 宿主在沙箱中执行断言
    test_ok = run_tests(patched_code)
    return test_ok, patched_code

bug_fixed, final_code = run_coding_agent_loop()
print("=== 实验 3 实测结果（Coding Agent 自动闭环）===")
print(f"基线代码测试通过 : {run_tests(SOURCE_CODE_BUGGY)} (成功捕获错误)")
print(f"Agent 修复后测试  : {bug_fixed} (单元测试全部通过！)")


In [ ]:
# [Cell 5: 自动验收与课程回执生成]
assert res_outcome["total_tokens"] < res_fine["total_tokens"] * 0.5, "实验 1 验收失败"
assert tokens_lazy < tokens_eager * 0.2, "实验 2 验收失败"
assert bug_fixed, "实验 3 验收失败"

receipt = {
    "lesson": "lesson-08-tools-mcp-coding-agent",
    "status": "PASSED",
    "metrics": {
        "exp1_token_reduction_pct": round((1 - res_outcome["total_tokens"]/res_fine["total_tokens"])*100, 1),
        "exp2_context_saved_pct": round((1 - tokens_lazy / tokens_eager)*100, 1),
        "exp3_coding_closed_loop_verified": True
    }
}
print("🎉 恭喜！第 8 课全部 3 个实验验收通过！成绩回执如下：")
print(json.dumps(receipt, indent=2, ensure_ascii=False))
